In [143]:
import pandas as pd
import numpy as np

In [144]:
bmp_ump = pd. read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\BMP_UMP.xlsx")

In [145]:
kam_nkam = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\KAM_NKAM.xlsx")

In [146]:
pipeline = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Display\Pipeline Report.xlsx",sheet_name="Line Item Export")

In [147]:
billing = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Display\Billing Report.xlsx",sheet_name="LineItem Report")

In [148]:
search_billing = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Display\Search Billing Sign off Jan'26 - Accruals - Final Billing Sign off.csv",skiprows=1,skipfooter=9,engine='python')

In [149]:
advance_report = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Display\Advance Report.csv",skiprows = 2)

In [150]:
master_classification = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Master_classification.xlsx")

In [151]:
Tagging = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Tagging.xlsx")

In [152]:
supercat_wise = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Supercat wise.xlsx")

In [153]:
# supercat_wise.columns

In [154]:
kam_nkam = kam_nkam.rename(columns={"Concatenate":"BUxBrand"})

- Function to Convert Object to INT64

In [155]:
# def obj2int(series):
#     return (
#         series.astype(str)
#               .str.replace(',', '', regex=False)
#               .astype(float)
#               .round(0)
#               .astype('Int64')
#     )

def obj2int(series):
    return (
        pd.to_numeric(
            series.astype(str).str.replace(',', '', regex=False), 
            errors='coerce'
        )
        .round(0)
        .astype('Int64')
    )


In [156]:
def flt2int(series):

    if series.dtype == 'object':
        series = series.str.replace(',', '', regex=False)

    return (
        pd.to_numeric(series, errors='coerce') # Invalid strings become NaN
        .round(0)
        .astype('Int64')
    )


# Pipeline Report


In [157]:
# pipeline.info()

In [158]:
# advance.info()

In [159]:
# billing.info()

- Clean pipeline Data


In [160]:
pipeline["SuperCatagory"] = (
    pipeline["Super Category"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

In [161]:
pipeline['Line Item Net Budget'].sum()

np.float64(1438057048.3200002)

- Date Check

In [162]:
StartOfMonth = '2026-01-01'
pipeline['Date Check'] = pipeline['Line Item Start Date'] < StartOfMonth

pipeline['Date Check'].value_counts()

Date Check
False    2092
True      189
Name: count, dtype: int64

- Days

In [163]:
Pipeline_Date = pd.to_datetime('2026-01-14')

In [164]:
diff1 = (Pipeline_Date - pipeline['Line Item Start Date']).dt.days
diff2 = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days

#It ensures that any negative results are reset to zero.

pipeline['Days'] = np.minimum(diff1, diff2).clip(0)


In [165]:
pipeline['Days'].sum()

np.int64(16801)

- Consumption

In [166]:
duration = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days.clip(lower=1)

pipeline['Consumption'] = np.where(
    pipeline['Date Check'].astype(str) == "True",
    0,
    (pipeline['Line Item Net Budget'] / duration) * pipeline['Days']
)


In [167]:
pipeline['Consumption'].sum()

np.float64(169451742.57024616)

In [168]:
Exception_map = [157770,157771,157773,157774]
pipeline['Exception'] = np.where(pipeline['Line Item Id'].isin(Exception_map), 'Exception', '0')

- Exception

In [169]:
pipeline['Exception'].value_counts()

Exception
0    2281
Name: count, dtype: int64

- Estimates

In [170]:
pipeline['Final_Estimated'] = np.where(pipeline['Exception'] == "Exception", 0, np.where(pipeline['Date Check'] == True, 0, pipeline['Line Item Net Budget']))

In [171]:
pipeline['Final_Estimated'].sum()

np.float64(1087322639.03)

- Rate Card Bifurcation

In [172]:
rateCard_map = dict(zip(Tagging['Rate Card'],Tagging['Type_RC']))
RC_map = dict(zip(Tagging['RC_type_Tag'],Tagging['RC_Tag']))
pipeline['Rate Card Bifurcation'] = pipeline['Rate Card Dimension'].map(rateCard_map).fillna(pipeline['Rate Card Dimension'].map(RC_map)).fillna(0)

In [173]:
# pipeline['Rate Card Bifurcation'].value_counts()

 - Alpha/MP

In [174]:
pipeline['Alpha_MP_Tag'] = np.where(pipeline['SuperCatagory']=="Quich","HL",np.where(pipeline['Line Item Business Type']=="",pipeline['RO Business Type'],pipeline['Line Item Business Type']))

In [175]:
# pipeline['Alpha_MP_Tag'].value_counts()

In [176]:
pipeline['New Alpha/MP'] = pipeline['Alpha_MP_Tag'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


In [177]:
# pipeline['New Alpha/MP'].value_counts()

- New Supercategory

In [178]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (pipeline['SuperCatagory'] == "Quick DBEFM"),(pipeline['SuperCatagory'] == "Quick Grocery"),
    (pipeline['SuperCatagory'] == "Quick Staples"),(pipeline['SuperCatagory'] == "Quick Healthcare"),
    (pipeline['SuperCatagory'] == "Quick Home"),(pipeline['SuperCatagory'] == "Quick Foods"),
    (pipeline['SuperCatagory'].str.contains("Quick", na=False))
             ]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
              pipeline['RO Brand'].map(map_sheet1).fillna(0)
          ]

default_lookup = pipeline['SuperCatagory'].map(map_mc_b) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_c)) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

pipeline['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [179]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
pipeline['New BU'] = pipeline['New Supercat'].map(bu_mapping).fillna(
    pipeline['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

In [180]:
# pipeline['New BU'].value_counts()

- RO Start Date

In [181]:

pipeline = pipeline.reset_index(drop=True)

date_mapping = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Start Date']

pipeline['RO Start Date'] = pipeline['Line Item Id'].map(date_mapping).fillna(pd.to_datetime('1900-01-01'))


- Check

In [182]:
last_month_end = '2025-12-31'
pipeline['Check'] = np.where(pipeline['RO End Date'] >= pipeline['RO Start Date'] ,"Yes","No")

In [183]:
# pipeline['Check'].value_counts()

- RO Amount

In [184]:

sumifs_q = pipeline.groupby('RO Number')['Line Item Net Budget'].transform('sum')

billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Amount']
xlookup_val = pipeline['Line Item Id'].map(billing_map).fillna(0)

SOM_date = pd.to_datetime('2026-01-01')

condition = (pipeline['Check'] == "Yes") & (pipeline['RO Start Date'] >= SOM_date)

calculation = np.divide(xlookup_val, sumifs_q, out=np.zeros_like(xlookup_val), where=sumifs_q!=0) * pipeline['Line Item Net Budget']


pipeline['RO Amount'] = np.where(condition, calculation, 0)


In [185]:
pipeline['RO Amount'].sum()

np.float64(976427772.0144452)

In [186]:
pipeline[pipeline['Check'] == "Yes"] ['RO Amount'].sum()

np.float64(976427772.0144452)

In [187]:
pipeline['SCxBrandxAlpha_MP'] = pipeline['New Supercat'].astype(str) + pipeline['RO Brand'].astype(str) + pipeline['New Alpha/MP'].astype(str)
PB_mapping = Tagging.drop_duplicates('SCxBrandxType').set_index('SCxBrandxType')['PrivateBrand']
pipeline['PrivateBrand'] = pipeline['SCxBrandxAlpha_MP'].map(PB_mapping).fillna("0")

In [188]:
# pipeline['PrivateBrand'].value_counts()

- BMP/UMP

In [189]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [190]:
pipeline['BMP_UMP'] = pipeline.apply(BMP_NonBMP, axis=1)

In [191]:
pipeline['BMP_UMP'].value_counts()

BMP_UMP
Non BMP    2168
BMP         113
Name: count, dtype: int64

- KAM/NKAM

In [192]:
pipeline['BUxBrand'] = pipeline['New BU'].astype(str).str.upper().str.strip() + pipeline['RO Brand'].astype(str).str.upper().str.strip()

In [193]:
kam_nkam['BUxBrand'] = kam_nkam['BUxBrand'].astype(str).str.upper().str.strip()
kam_nkam['Brand-1'] = kam_nkam['Brand-1'].astype(str).str.upper().str.strip()
kam_nkam['Owner'] = kam_nkam['Owner'].astype(str).str.upper().str.strip()
pipeline['BUxBrand'] = pipeline['BUxBrand'].astype(str).str.upper().str.strip()
pipeline['Brand'] = pipeline['Brand'].astype(str).str.upper().str.strip()

mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

BUxBrand_KN = pipeline['BUxBrand'].map(mapping_BUXBrand)
Brand_KN = pipeline['Brand'].map(mapping_Brand)

conditions = [ (pipeline['New Supercat'] == "Automobile"), BUxBrand_KN.notna(), Brand_KN.notna() ]

choices = [ "KAM", BUxBrand_KN, Brand_KN ]

pipeline['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [194]:
pipeline['kam_nkam'].value_counts()

kam_nkam
KAM     1618
NKAM     663
Name: count, dtype: int64

In [195]:
pipeline[pipeline['kam_nkam'] == "KAM"]['Final_Estimated'].sum()

np.float64(706605843.09)

# Search Billing Sign Off


- Convert object to integer

In [196]:
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['LineItem Gross Budget'] = obj2int(search_billing['LineItem Gross Budget'])
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['Gross ad spend'] = obj2int(search_billing['Gross ad spend'])
search_billing['LineItem Net Budget'] = obj2int(search_billing['LineItem Net Budget'])

- Clean supercategory

In [197]:
search_billing["SuperCatagory"] = (
    search_billing["Industry.1"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

- New Super Category

In [198]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (search_billing['SuperCatagory'] == "Quick DBEFM"),(search_billing['SuperCatagory'] == "Quick Grocery"),
    (search_billing['SuperCatagory'] == "Quick Staples"),(search_billing['SuperCatagory'] == "Quick Healthcare"),
    (search_billing['SuperCatagory'] == "Quick Home"),(search_billing['SuperCatagory'] == "Quick Foods"),
    (search_billing['SuperCatagory'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
    search_billing['RO Brand'].map(map_sheet1).fillna(0)
]

default_lookup = search_billing['SuperCatagory'].map(map_mc_b) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_c)) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

search_billing['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [199]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
search_billing['New BU'] = search_billing['New Supercat'].map(bu_mapping).fillna(
    search_billing['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

In [200]:
# search_billing['New Supercat'].value_counts()

- BMP/UMP


In [201]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [202]:
search_billing['BMP_UMP'] = search_billing.apply(BMP_NonBMP, axis=1)

In [203]:
# search_billing['BMP_UMP'].value_counts()

In [204]:
search_billing["BUxBrand_Cleaned"] = search_billing["New BU"].astype(str).str.upper().str.strip() + search_billing["RO Brand"].astype(str).str.upper().str.strip()

In [205]:
BUxBrand_map =dict(zip(kam_nkam['BUxBrand'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
Brand_map = dict(zip(kam_nkam['Brand-1'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
search_billing['KAM_NKAM_check'] = search_billing['BUxBrand_Cleaned'].map(BUxBrand_map).fillna(search_billing['RO Brand'].astype(str).str.upper().str.strip().map(Brand_map)).fillna("KAM")

In [206]:
search_billing['KAM_NKAM_check'].value_counts()

KAM_NKAM_check
KAM     483
NKAM     78
Name: count, dtype: int64

In [207]:
# search_billing[search_billing["KAM_NKAM_check"]=="NKAM"]['Estimate'].sum()

- BOC/BOL


In [208]:
BOL_BOC_map = dict(zip(Tagging['Type_Dimention'],Tagging['Check']))
search_billing['BOC_BOL'] = search_billing['Billing Details'].map(BOL_BOC_map).fillna(0)

In [209]:
search_billing['BOC_BOL'].value_counts()

BOC_BOL
BOC    321
BOL    180
NO      60
Name: count, dtype: int64

- Consumption

In [210]:
search_billing["Consumption"] = np.where(search_billing['BOC_BOL']=="NO",0,
      search_billing[['LineItem Net Budget','Gross ad spend','Net Billing Number']].min(axis=1))

In [211]:
search_billing["Consumption"].sum()

np.int64(73134683)

- Estimates

In [212]:
Consumed_Days = 12   
Total_Days = 31  

gross_spend = search_billing['Gross ad spend'].astype(float)
gross_budget = search_billing['LineItem Gross Budget'].astype(float)

spend_ratio = (gross_spend / gross_budget).fillna(0)
spend_ratio = spend_ratio.replace([np.inf, -np.inf], 0)

conditions = [
    (search_billing['BOC_BOL'] == "NO"),
    (search_billing['BOC_BOL'] == "BOL") & (spend_ratio > 0.4)
]

choices = [ 0, search_billing['Net Billing Number'] ]

calc_weighted = (gross_spend / Consumed_Days) * Total_Days
net_budget = search_billing['LineItem Net Budget'].astype(float)
net_billing_num = search_billing['Net Billing Number'].astype(float)
default_case = (
    calc_weighted
    .clip(upper=net_budget)      # Compare weighted vs Net Budget
    .clip(upper=net_billing_num) # Compare result vs Net Billing Number
)

search_billing['Estimate'] = np.select(conditions, choices, default=default_case)

search_billing['Estimate'] = pd.to_numeric(search_billing['Estimate'], errors='coerce').round(0).astype('Int64')


In [213]:
search_billing['Estimate'].sum()

np.int64(103433635)

In [214]:
# search_billing.info(verbose=True,show_counts=True)

- Cons. As a % of Budget

In [215]:
search_billing["Cons. As a % of Budget"] = (search_billing['Consumption']/search_billing['Estimate']).fillna(0)

- Cons. As a % of Net Billing

In [216]:
search_billing["Cons. As a % of Net Billing"] = (search_billing['Consumption']/search_billing['Net Billing Number']).fillna(0)

- Alpha/MP

In [217]:
lookup_map = Tagging.set_index('Supercat')['Alpha/MP'].to_dict()

In [218]:
# Alpha_MP
cond_quick = search_billing['SuperCatagory'].str.contains("Quick", na=False, case=False)

cond_lookup = search_billing['New Supercat'].isin(lookup_map.keys())

choices = [
    "HL",
    search_billing['New Supercat'].map(lookup_map)
]

default_logic = np.select(
    [search_billing['Line Item Business Type'].isna() | (search_billing['Line Item Business Type'] == ""), (search_billing['Line Item Business Type'] == "BMP")],
    [search_billing['RO Business Type'], "MP"],
    default=search_billing['Line Item Business Type']
)

search_billing['Alpha_MP'] = np.select([cond_quick, cond_lookup],
                                        choices,
                                        default=default_logic)


In [219]:
search_billing['Alpha_MP'].value_counts()

Alpha_MP
MP                 301
Alpha              203
HL                  55
Advance Payment      1
Managed HPMU         1
Name: count, dtype: int64

- KAM/NKAM

In [220]:
search_billing["BUxBrand"] = search_billing["New BU"].astype(str) + search_billing["RO Brand"].astype(str)

In [221]:
mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

SBUxBrand_KN = search_billing['BUxBrand'].map(mapping_BUXBrand)
SBrand_KN = search_billing['RO Brand'].map(mapping_Brand)

conditions = [ (search_billing['New Supercat'] == "Automobile"), SBUxBrand_KN.notna(), SBrand_KN.notna() ]

choices = [ "KAM", SBUxBrand_KN, SBrand_KN ]

search_billing['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [222]:
search_billing['kam_nkam'].value_counts()

kam_nkam
KAM     545
NKAM     16
Name: count, dtype: int64

In [223]:
search_billing[search_billing['SuperCatagory'] == "Automobile"]['New Supercat'].value_counts()

Series([], Name: count, dtype: int64)

In [224]:
# billing.info()

# Advance Report


In [225]:
advance = advance_report.copy()

In [226]:
billing['Line Item Id'] = flt2int(billing['Line Item Id'])

In [227]:
billing['Line Item Id'].dtype

Int64Dtype()

- Line Item Id


In [228]:
advance.loc[:, 'Line Item Id'] = (
    advance["Name"].str.extract(r"(\d+)")
    .fillna(0)
    .astype(int)
)

In [229]:
advance['Line Item Id'].count()

np.int64(1190)

In [230]:
# advance[advance['Line Item Id'] == 0]

- Supercat

In [231]:
map_p1 = pipeline.set_index('Line Item Name')['SuperCatagory'].to_dict()
map_p2 = pipeline.set_index('Line Item Id')['SuperCatagory'].to_dict()
map_b3 = billing.set_index('Line Item Id')['Industry.1'].to_dict()

advance['Supercat'] = advance['Name'].map(map_p1)

advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_p2))
advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_b3))

advance['Supercat'] = advance['Supercat'].fillna(0)


In [232]:
# advance['Supercat'].value_counts()

- Brand


In [233]:
map_m_to_b = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['RO Brand']
map_l_to_b = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Brand']

advance['Brand'] = (
    advance['Name'].map(map_m_to_b)
    .fillna(advance['Line Item Id'].map(map_l_to_b))
    .fillna(0)
)


In [234]:
advance['Brand'] = advance['Brand'].str.upper()

In [235]:
Tagging['Brand Name'] = Tagging['Brand Name'].str.upper()

- New SuperCat

In [236]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_ba = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_ca = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df_a = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_ea = temp_df.index.to_series()

conditions = [
    (advance['Supercat'] == "Quick DBEFM"),(advance['Supercat'] == "Quick Grocery"),
    (advance['Supercat'] == "Quick Staples"),(advance['Supercat'] == "Quick Healthcare"),
    (advance['Supercat'] == "Quick Home"),(advance['Supercat'] == "Quick Foods"),
    (advance['Supercat'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition", 
    advance['Brand'].map(map_sheet1).fillna(0)
]

default_lookup = advance['Supercat'].map(map_mc_b) \
                 .fillna(advance['Supercat'].map(map_mc_c)) \
                 .fillna(advance['Supercat'].map(map_mc_e)) \
                 .fillna(0)

advance['New Supercat'] = np.select(conditions, choices, default=default_lookup)


In [237]:
# advance[advance['New Supercat'] == 0]['Brand']

- New BU


In [238]:
master_map = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')['BU']

advance['New BU'] = advance['New Supercat'].map(master_map).fillna(advance['New Supercat'].replace({"Staples": "Grocery"}))


- Alpha/MP

In [239]:
map_sheet1_no = Tagging.set_index('Supercat')['Alpha/MP'].to_dict()
map_pipe_mae = pipeline.set_index('Line Item Name')['RO Business Type'].to_dict()
map_pipe_lae = pipeline.set_index('Line Item Id')['RO Business Type'].to_dict()

is_quick = advance['Supercat'].str.contains("Quick", case=False, na=False)

is_in_sheet1 = advance['New Supercat'].isin(Tagging['Supercat'])

pipeline_lookup = (
    advance['Name'].map(map_pipe_mae)
    .fillna(advance['Line Item Id'].map(map_pipe_lae))
    .fillna(0)
)

conditions = [
    is_quick,
    is_in_sheet1
]

choices = [
    "HL",
    advance['New Supercat'].map(map_sheet1_no),
]

advance['Alpha_MP'] = np.select(conditions, choices, default=pipeline_lookup)


In [240]:
advance['Alpha_MP'].value_counts()

Alpha_MP
Alpha    773
HL       170
MP       102
0         79
BMP       41
3P        25
Name: count, dtype: int64

- New Alpha/MP

In [241]:
advance['New Alpha/MP'] = advance['Alpha_MP'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


In [242]:
advance['New Alpha/MP'].value_counts()

New Alpha/MP
Alpha    773
HL       170
MP       143
0         79
3P        25
Name: count, dtype: int64

- Line Item Net Budget

In [243]:
map_p_m_q = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Net Budget']
map_p_l_q = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Net Budget']
map_b_k_t = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['LineItem Net Budget']

advance['Line Item Net Budget'] = (
    advance['Name'].map(map_p_m_q)
    .fillna(advance['Line Item Id'].map(map_p_l_q))
    .fillna(advance['Line Item Id'].map(map_b_k_t))
    .fillna(0)
)


In [244]:
advance['Line Item Net Budget'].sum()

np.float64(362362041.35)

Check (Supercat Match)

In [245]:
advance['Check'] = advance['New Supercat'].isin(supercat_wise['Super Categories'])


In [246]:
# advance['Check'].value_counts()

- Start Date


In [247]:
map_m_to_o = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Start Date']
map_l_to_o = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Start Date']

advance['Start_Date'] = (
    advance['Name'].map(map_m_to_o)
    .fillna(advance['Line Item Id'].map(map_l_to_o))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [248]:
# advance['Start_Date'].head()

- End Date

In [249]:
map_m_to_p = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item End Date']
map_l_to_p = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item End Date']

advance['End_Date'] = (
    advance['Name'].map(map_m_to_p)
    .fillna(advance['Line Item Id'].map(map_l_to_p))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [250]:
# advance['End_Date'].head()

- CPD/CMP

In [251]:
map_m_to_x = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Rate Card']
map_l_to_x = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Rate Card']

advance['CPD_CPM'] = (
    advance['Name'].map(map_m_to_x)
    .fillna(advance['Line Item Id'].map(map_l_to_x))
    .fillna(0)
)

In [252]:
advance['CPD_CPM'].value_counts()

CPD_CPM
vCPM                            692
Less than 100 SOV               219
100 SOV - CPD                   102
0                                79
100 SOV                          32
vCPM Rate Card                   23
100 SOV - inactive               19
less than 100 SOV - inactive      8
CPC                               8
less than 100 SOV                 7
Less Than 100 SOV                 1
Name: count, dtype: int64

- BOL/BOC

In [253]:
bol_list = ["Less than 100 SOV", "CPD", "CPT", "Reservation Buy", "100 SOV - CPD", "100 SOV" ]
advance['BOL/BOC'] = np.where(advance['CPD_CPM'].str.strip().isin(bol_list), "BOL", "BOC")

In [254]:
advance['BOL/BOC'].value_counts()

BOL/BOC
BOC    837
BOL    353
Name: count, dtype: int64

- Date_Check


In [255]:
SOM_AR = pd.to_datetime('2026-01-01')
advance['Date_Check'] = np.where(advance['BOL/BOC'] == 'BOC', "False", advance['Start_Date'] < SOM_AR)

In [256]:
advance['Date_Check'].value_counts()

Date_Check
False    1189
True        1
Name: count, dtype: int64

- Date of Report

In [257]:
advance['Date of Report'] = pd.to_datetime('2026-01-14')

- No of Days


In [258]:
advance['NoofDays'] = np.maximum(
    np.minimum(
        (advance["Date of Report"] - advance["Start_Date"]).dt.days,
        (advance["End_Date"] - advance["Start_Date"]).dt.days
    ),
    1
)

In [259]:
# advance['NoofDays'].sum()

- Final Consumption Numbers

In [260]:
duration = (advance['End_Date'] - advance['Start_Date']).dt.days.clip(lower=1)

conditions = [
    (advance['Date_Check'] == "True"),
    (advance['BOL/BOC'] == "BOL")
]

choices = [
    0,
    (advance['Line Item Net Budget'] / duration) * advance['NoofDays'],
]

advance['Final Consumption Numbers'] = np.select(
    conditions,
    choices,
    default=np.minimum(advance['Net Spend'], advance['Line Item Net Budget'])
)


In [261]:
advance['Final Consumption Numbers'].sum()

np.float64(234675589.05791587)

- Final Estimates

In [262]:
Bool_val = True
TotalDays = 31
ConsumedDays = 14

conditions = [(advance['Date_Check'] == Bool_val), (advance['BOL/BOC'] == "BOL") ]
choices = [ 0, advance['Line Item Net Budget'] ]

advance['Final Estimates'] = np.select( conditions, choices,
    default=np.minimum(
        ((advance['Net Spend'] * TotalDays) / ConsumedDays),
        advance['Line Item Net Budget']
    )
)


In [263]:
advance['Final Estimates'].sum()

np.float64(303668383.23422146)

- Private brand


In [264]:
advance['LookupKey'] = advance['New Supercat'].astype(str) + advance['Brand'].astype(str) + advance['New Alpha/MP'].astype(str)
Tagging['LookupKey'] = Tagging['Supercat_PB'].astype(str) + Tagging['Brand_PB'].astype(str) + Tagging['Type_PB'].astype(str)

tagging_map = Tagging.drop_duplicates('LookupKey').set_index('LookupKey')['PrivateBrand']

advance['Private Brand'] = advance['LookupKey'].map(tagging_map)


In [265]:
# advance['Private Brand'].value_counts()

- KAM/NKAM


In [266]:
map_BUxBrand = kam_nkam.drop_duplicates('BUxBrand').set_index('BUxBrand')['Owner']
map_Brand = kam_nkam.drop_duplicates('Brand-1').set_index('Brand-1')['Owner']
advance['BUxBrand'] = advance['New BU'].astype(str) + advance['Brand'].astype(str)

condition = (advance['New Supercat'] == "Automobile")

BUxBrand_mapped = advance['BUxBrand'].map(map_BUxBrand).fillna("KAM")

final_map = advance['Brand'].map(map_Brand).fillna(BUxBrand_mapped)

conditions = [condition]
choices = ["KAM"]

advance['KAM/NKAM'] = np.select(conditions, choices, default=final_map)


In [267]:
advance[advance['KAM/NKAM'] == "KAM"]['Final Estimates'].sum()

np.float64(228725158.50779286)

- BMP/UMP

In [268]:
advance['BMP/UMP'] = np.where(advance['New Supercat'] == "Automobile", "BMP",np.where(advance['Alpha_MP'] == "BMP", "BMP","NonBMP"))

In [269]:
# advance['BMP/UMP'].value_counts()

- Internal RO Number

In [270]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal RO number']

advance['Internal RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)


- External RO NO

In [271]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['External RO number']

advance['External RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Advertiser

In [272]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Advertiser']

advance['Advertiser'] = advance['Line Item Id'].map(billing_map).fillna(0)

- PAN Number

In [273]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Pan Card Number']

advance['PAN Number'] = advance['Line Item Id'].map(billing_map).fillna(0)

- GSTIN

In [274]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['GST']

advance['GSTIN'] = advance['Line Item Id'].map(billing_map).fillna(0)

- RO Link

In [275]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Link']

advance['RO Link'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Internal Additional Document

In [276]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal Additional Document']

advance['Additional Document'] = advance['Line Item Id'].map(billing_map).fillna(0)

- loaded vs Estimates

In [277]:
advance['Loaded Vs Estimates'] = advance['Line Item Net Budget'] - advance['Final Estimates']

In [278]:
advance['Loaded Vs Estimates'].sum()

np.float64(58693658.115778565)

- Net Spend Vs Estimates

In [279]:
advance['NetSpend vs Estimates'] = advance['Line Item Net Budget'] - advance['Net Spend']

In [280]:
advance['NetSpend vs Estimates'].sum()

np.float64(143026111.8567)

In [281]:
ad_grp = advance.groupby(['New BU','New Alpha/MP'])['Final Consumption Numbers'].sum()

In [282]:
# print(ad_grp)

In [283]:
advance.to_excel("advance_report.xlsx",index=False)